# Cross-Domain Anchor Trajectories for CDAD-Planner

Этот ноутбук готовит **нормированные якорные траектории** для будущей модели **Cross-Domain Anchor Diffusion Planner (CDAD-Planner)**.

## Зачем нужен этот ноутбук

Цель статьи — не копировать DiffusionDrive напрямую, а построить компактную diffusion-based модель, которая работает с двумя разными embodied-доменами:

1. **Waymo / vehicle domain** — фронтальная камера автомобиля и будущая ego-траектория.
2. **i2Nav-Robot / robot domain** — фронтальная/левая камера наземного робота и logged future trajectory.

У автомобиля и робота разные физические масштабы, кинематика и среда. Поэтому напрямую смешивать physical waypoints `(x, y, yaw)` неудобно. Вместо этого мы приводим оба домена к единому формату:

```text
RGB image → image-space future trajectory
```

То есть каждая trajectory хранится как набор точек в координатах изображения:

```text
[(u1, v1), (u2, v2), ..., (uM, vM)]
```

Затем точки нормируются в диапазон:

```text
u_norm = 2 * u / W - 1
v_norm = 2 * v / H - 1
```

И итоговое пространство становится единым для машин и роботов:

```text
trajectory_uv_norm ∈ [-1, 1]^(M×2)
```

## Идея CDAD-Planner

CDAD-Planner будет использовать **shared cross-domain trajectory anchor vocabulary**.

Схема будущей модели:

```text
single RGB image
    ↓
image encoder
    ↓
anchor classifier
    ↓
top-k trajectory anchors
    ↓
conditional diffusion / flow refinement
    ↓
final image-space future trajectory
```

Вместо генерации траектории из чистого шума модель стартует от одного из заранее найденных якорей. Это должно сделать модель:

- меньше;
- быстрее;
- стабильнее в small-data regime;
- ближе по идее к DiffusionDrive, но не копией;
- пригодной для mixed Waymo + i2Nav training.

## Что делает этот ноутбук

1. Загружает Waymo corridor dataset, созданный через `build_corridor_dataset_filtered_front_logged-3.ipynb`.
2. Загружает i2Nav-Robot corridor dataset `N10 / 2 sec`.
3. Берет subset для статьи: примерно `10k` Waymo + все доступные i2Nav N10 samples.
4. Извлекает vector trajectory для каждого sample.
5. Если в manifest нет готовых `uv`-точек, восстанавливает траекторию из `centerline_mask` / `corridor_mask`.
6. Ресемплирует каждую траекторию в фиксированное число точек `M=8`.
7. Нормирует trajectory points в `[-1, 1]`.
8. Строит KMeans anchors для нескольких вариантов `K`:

```text
K = 8, 12, 16, 20, 24, 32
```

9. Основной вариант для статьи сохраняет `K=20`.
10. Сохраняет:

```text
cross_domain_anchor_trajectories_cdad_planner_v1/
  trajectory_vectors_combined_train.jsonl
  anchor_k_selection_metrics.json
  anchors_K20_M8_norm.npy
  anchors_K20_M8_norm.json
  trajectory_vectors_with_anchor_K20.jsonl
  anchor_K20_domain_distribution.json
  anchors_K20_preview.jpg
  anchor_generation_summary.json
```

## Почему KMeans через OpenCV

В этой среде `scikit-learn` падает из-за отсутствующего `scipy`:

```text
ModuleNotFoundError: No module named 'scipy'
```

Поэтому кластеризация в этом ноутбуке сделана через:

```python
cv2.kmeans
```

Это убирает зависимость от `sklearn/scipy`. Все остальное в логике ноутбука сохранено.

## Как выбирать K для статьи

Мы не фиксируем `K=20` вслепую. Ноутбук считает anchors для нескольких `K` и сохраняет метрики.

Выбор `K=20` можно будет аргументировать так:

1. `K=20` дает достаточно разнообразные trajectory modes: straight, left, right, soft curves, sharper turns.
2. При росте `K` после 20 выигрыш по `MSE/RMSE` обычно становится менее существенным.
3. `K=20` остается компактным для маленького diffusion/flow-refinement planner.
4. Визуально anchors не выглядят чрезмерно раздробленными.
5. Размеры кластеров и domain distribution можно проверить через сохраненные таблицы.

## Важно

Этот ноутбук готовит именно **anchor vocabulary** и **vector trajectory assignments**. Он еще не обучает CDAD-Planner. Следующий этап после этого ноутбука:

```text
anchors + trajectory_vectors_with_anchor_K20.jsonl
    ↓
image encoder + anchor classifier + conditional diffusion/flow refiner
```


In [ ]:
from pathlib import Path
import json
import random
import math
import time
from collections import Counter, defaultdict

import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display

try:
    import cv2
    print("cv2: OK", cv2.__version__)
except Exception as e:
    raise RuntimeError("cv2 is required for mask-based trajectory extraction and KMeans anchors") from e

PROJECT_ROOT = Path("/home/Jupyter/datasets/tesla/Waymo_open_dataset")

# Waymo dataset discovered from train_corridor_resnet50_augmented_no_intent_v2_pbar-2.ipynb:
# DATASET_ROOT = PROJECT_ROOT / "prepared_camera_corridor_dataset_filtered_front_v1"
WAYMO_ROOT = PROJECT_ROOT / "prepared_camera_corridor_dataset_filtered_front_v1"
WAYMO_CAMERA_NAME = "FRONT"
WAYMO_TRAIN_MANIFEST = WAYMO_ROOT / "train" / WAYMO_CAMERA_NAME / "manifest.jsonl"
WAYMO_VAL_MANIFEST = WAYMO_ROOT / "val" / WAYMO_CAMERA_NAME / "manifest.jsonl"

# Current i2Nav N10 result. Note: folder name is historical v1_2_sec, but config inside says N10.
I2NAV_ROOT = PROJECT_ROOT / "prepared_i2nav_robot_corridor_dataset_v1_2_sec"
I2NAV_MANIFEST = I2NAV_ROOT / "meta" / "manifest.jsonl"

OUT_DIR = PROJECT_ROOT / "cross_domain_anchor_trajectories_cdad_planner_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Article-oriented configuration
SEED = 42
M_POINTS = 8
K_LIST = [8, 12, 16, 20, 24, 32]
MAIN_K = 20

# Use approximately the planned article subset.
WAYMO_MAX_SAMPLES = 10_000
I2NAV_MAX_SAMPLES = None  # None = all i2Nav rows

# Use only training split for anchor fitting by default to avoid test leakage.
FIT_SPLITS = {"train"}

# Mask extraction
MASK_THRESHOLD = 10
MIN_COMPONENT_PIXELS = 30
MIN_UNIQUE_Y = 8

random.seed(SEED)
np.random.seed(SEED)
cv2.setRNGSeed(SEED)

print("PROJECT_ROOT:", PROJECT_ROOT, PROJECT_ROOT.exists())
print("WAYMO_ROOT:", WAYMO_ROOT, WAYMO_ROOT.exists())
print("WAYMO_TRAIN_MANIFEST:", WAYMO_TRAIN_MANIFEST, WAYMO_TRAIN_MANIFEST.exists())
print("WAYMO_VAL_MANIFEST:", WAYMO_VAL_MANIFEST, WAYMO_VAL_MANIFEST.exists())
print("I2NAV_ROOT:", I2NAV_ROOT, I2NAV_ROOT.exists())
print("I2NAV_MANIFEST:", I2NAV_MANIFEST, I2NAV_MANIFEST.exists())
print("OUT_DIR:", OUT_DIR)


In [ ]:
def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                # Safe for manifests that were once written live.
                continue
    return rows

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def resolve_path(dataset_root, rel_or_abs):
    if rel_or_abs is None:
        return None
    p = Path(rel_or_abs)
    if p.is_absolute():
        return p
    candidates = [
        Path(dataset_root) / p,
        PROJECT_ROOT / p,
    ]
    for c in candidates:
        if c.exists():
            return c
    return Path(dataset_root) / p

def find_first_existing_key(row, keys):
    for k in keys:
        if k in row and row[k]:
            return k, row[k]
    return None, None

IMAGE_KEYS = ["image_path", "rgb_path", "camera_image_path", "image", "img_path"]
MASK_KEYS_PRIORITY = [
    "centerline_mask_path",
    "corridor_mask_path",
    "mask_path",
    "target_path",
    "label_path",
    "mask",
]

VECTOR_KEYS = [
    "trajectory_uv",
    "future_uv",
    "future_points_uv",
    "projected_points",
    "trajectory_points",
    "waypoints_uv",
    "future_waypoints_uv",
]

def get_image_path(row, dataset_root):
    key, value = find_first_existing_key(row, IMAGE_KEYS)
    return resolve_path(dataset_root, value), key

def get_mask_path(row, dataset_root):
    key, value = find_first_existing_key(row, MASK_KEYS_PRIORITY)
    return resolve_path(dataset_root, value), key

def try_get_vector_points(row):
    for key in VECTOR_KEYS:
        if key in row and row[key] is not None:
            arr = np.asarray(row[key], dtype=np.float32)
            if arr.ndim == 2 and arr.shape[1] >= 2 and len(arr) >= 2:
                return arr[:, :2], key
    return None, None


In [ ]:
# =========================
# Load Waymo + i2Nav manifests
# =========================

waymo_train = read_jsonl(WAYMO_TRAIN_MANIFEST)
waymo_val = read_jsonl(WAYMO_VAL_MANIFEST)
i2nav_rows = read_jsonl(I2NAV_MANIFEST)

for r in waymo_train:
    r["_source_dataset"] = "Waymo"
    r["_dataset_root"] = str(WAYMO_ROOT)
    r["_manifest_split"] = r.get("split", "train")
for r in waymo_val:
    r["_source_dataset"] = "Waymo"
    r["_dataset_root"] = str(WAYMO_ROOT)
    r["_manifest_split"] = r.get("split", "val")

for r in i2nav_rows:
    r["_source_dataset"] = "i2Nav-Robot"
    r["_dataset_root"] = str(I2NAV_ROOT)
    r["_manifest_split"] = r.get("split", "unknown")

waymo_all = waymo_train + waymo_val

print("Waymo train rows:", len(waymo_train))
print("Waymo val rows:", len(waymo_val))
print("Waymo all rows:", len(waymo_all))
print("i2Nav rows:", len(i2nav_rows))

if waymo_all:
    print("\nWaymo first keys:")
    print(sorted(waymo_all[0].keys()))
    print(json.dumps(waymo_all[0], ensure_ascii=False, indent=2)[:2500])

if i2nav_rows:
    print("\ni2Nav first keys:")
    print(sorted(i2nav_rows[0].keys()))
    print(json.dumps(i2nav_rows[0], ensure_ascii=False, indent=2)[:2500])


In [ ]:
# =========================
# Select article subset
# =========================

def stable_sample(rows, max_n, seed=42):
    if max_n is None or len(rows) <= max_n:
        return list(rows)
    rng = random.Random(seed)
    idxs = list(range(len(rows)))
    rng.shuffle(idxs)
    idxs = sorted(idxs[:max_n])
    return [rows[i] for i in idxs]

# Fit anchors on train split only where available.
waymo_fit_pool = [r for r in waymo_all if r.get("_manifest_split", r.get("split")) in FIT_SPLITS or r.get("split") in FIT_SPLITS]
if not waymo_fit_pool:
    waymo_fit_pool = waymo_all

i2nav_fit_pool = [r for r in i2nav_rows if r.get("split") in FIT_SPLITS]
if not i2nav_fit_pool:
    i2nav_fit_pool = i2nav_rows

waymo_selected = stable_sample(waymo_fit_pool, WAYMO_MAX_SAMPLES, seed=SEED)
i2nav_selected = stable_sample(i2nav_fit_pool, I2NAV_MAX_SAMPLES, seed=SEED + 1)

selected_rows = waymo_selected + i2nav_selected

print("Selected Waymo:", len(waymo_selected), "from pool:", len(waymo_fit_pool))
print("Selected i2Nav:", len(i2nav_selected), "from pool:", len(i2nav_fit_pool))
print("Selected total:", len(selected_rows))

print("Selected source counts:", Counter(r["_source_dataset"] for r in selected_rows))
print("Waymo split counts:", Counter(r.get("_manifest_split", r.get("split")) for r in waymo_selected))
print("i2Nav split counts:", Counter(r.get("split") for r in i2nav_selected))


In [ ]:
# =========================
# Trajectory extraction from vector fields or masks
# =========================

def resample_polyline(points, m=M_POINTS):
    """
    Resample a 2D polyline to m points by arc length.
    points: [N,2]
    """
    pts = np.asarray(points, dtype=np.float32)
    if pts.ndim != 2 or pts.shape[1] != 2 or len(pts) < 2:
        return None

    # Remove duplicated consecutive points.
    keep = [0]
    for i in range(1, len(pts)):
        if np.linalg.norm(pts[i] - pts[keep[-1]]) > 1e-3:
            keep.append(i)
    pts = pts[keep]
    if len(pts) < 2:
        return None

    seg = np.linalg.norm(pts[1:] - pts[:-1], axis=1)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(cum[-1])
    if total < 1e-3:
        return None

    targets = np.linspace(0.0, total, m)
    out = []
    for t in targets:
        j = int(np.searchsorted(cum, t, side="right") - 1)
        j = max(0, min(j, len(seg) - 1))
        denom = max(seg[j], 1e-6)
        a = (t - cum[j]) / denom
        p = (1 - a) * pts[j] + a * pts[j + 1]
        out.append(p)
    return np.asarray(out, dtype=np.float32)

def normalize_uv(uv, width, height):
    uv = np.asarray(uv, dtype=np.float32)
    out = np.empty_like(uv, dtype=np.float32)
    out[:, 0] = 2.0 * uv[:, 0] / max(float(width - 1), 1.0) - 1.0
    out[:, 1] = 2.0 * uv[:, 1] / max(float(height - 1), 1.0) - 1.0
    return out

def denormalize_uv(uv_norm, width, height):
    uv_norm = np.asarray(uv_norm, dtype=np.float32)
    out = np.empty_like(uv_norm, dtype=np.float32)
    out[:, 0] = (uv_norm[:, 0] + 1.0) * 0.5 * max(float(width - 1), 1.0)
    out[:, 1] = (uv_norm[:, 1] + 1.0) * 0.5 * max(float(height - 1), 1.0)
    return out

def extract_polyline_from_mask(mask_path, threshold=MASK_THRESHOLD, m=M_POINTS):
    """
    Extract a coarse ordered trajectory from a binary centerline/corridor mask.

    Method:
    1. Threshold mask.
    2. Keep largest connected component.
    3. Order by image y from bottom to top.
    4. At M y-levels, take median x of component pixels near that y.
    5. Resample by arc length to M points.

    Output order: near-to-far in image coordinates, approximately bottom-to-top.
    """
    mask_path = Path(mask_path)
    gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return None, {"reason": "mask_read_failed"}

    h, w = gray.shape[:2]
    bw = (gray > threshold).astype(np.uint8)

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(bw, connectivity=8)
    if n_labels <= 1:
        return None, {"reason": "no_component", "width": w, "height": h}

    # exclude background label 0
    areas = stats[1:, cv2.CC_STAT_AREA]
    best_label = int(1 + np.argmax(areas))
    best_area = int(stats[best_label, cv2.CC_STAT_AREA])

    if best_area < MIN_COMPONENT_PIXELS:
        return None, {"reason": "small_component", "area": best_area, "width": w, "height": h}

    ys, xs = np.where(labels == best_label)
    if len(np.unique(ys)) < MIN_UNIQUE_Y:
        return None, {"reason": "too_few_unique_y", "unique_y": int(len(np.unique(ys))), "area": best_area, "width": w, "height": h}

    y_min, y_max = int(ys.min()), int(ys.max())
    y_levels = np.linspace(y_max, y_min, max(m * 3, 24))  # dense coarse line
    band = max(3, int(round(h * 0.006)))  # about 7 px for 1200p

    coarse = []
    for y0 in y_levels:
        dy = np.abs(ys.astype(np.float32) - float(y0))
        idx = np.where(dy <= band)[0]
        if len(idx) == 0:
            # fallback: nearest available y-row
            nearest_y = ys[int(np.argmin(dy))]
            idx = np.where(ys == nearest_y)[0]

        x_med = float(np.median(xs[idx]))
        y_med = float(np.median(ys[idx]))
        coarse.append([x_med, y_med])

    coarse = np.asarray(coarse, dtype=np.float32)

    # Remove duplicate points after coarse sampling.
    filtered = [coarse[0]]
    for p in coarse[1:]:
        if np.linalg.norm(p - filtered[-1]) > 1.0:
            filtered.append(p)
    coarse = np.asarray(filtered, dtype=np.float32)

    uv = resample_polyline(coarse, m=m)
    if uv is None:
        return None, {"reason": "resample_failed", "area": best_area, "width": w, "height": h}

    meta = {
        "reason": "ok",
        "area": best_area,
        "width": w,
        "height": h,
        "y_min": y_min,
        "y_max": y_max,
        "method": "mask_largest_component_y_median",
    }
    return uv, meta

def extract_trajectory_for_row(row, dataset_root, m=M_POINTS):
    """
    Prefer vector trajectory fields if present. Otherwise extract from mask.
    """
    img_path, img_key = get_image_path(row, dataset_root)
    mask_path, mask_key = get_mask_path(row, dataset_root)

    # Determine image size.
    width = row.get("image_width")
    height = row.get("image_height")
    if width is None or height is None:
        if img_path is not None and Path(img_path).exists():
            with Image.open(img_path) as im:
                width, height = im.size
        elif mask_path is not None and Path(mask_path).exists():
            with Image.open(mask_path) as im:
                width, height = im.size
        else:
            return None, {"reason": "no_image_or_mask_size"}

    width, height = int(width), int(height)

    uv_vec, vec_key = try_get_vector_points(row)
    if uv_vec is not None:
        uv = resample_polyline(uv_vec, m=m)
        if uv is None:
            return None, {"reason": "vector_resample_failed", "vector_key": vec_key}
        method = f"manifest_vector:{vec_key}"
        meta = {"reason": "ok", "method": method, "width": width, "height": height}
    else:
        if mask_path is None or not Path(mask_path).exists():
            return None, {"reason": "mask_missing", "mask_path": str(mask_path)}
        uv, meta = extract_polyline_from_mask(mask_path, m=m)
        if uv is None:
            meta["mask_path"] = str(mask_path)
            return None, meta
        method = meta.get("method", "mask")

    uv_norm = normalize_uv(uv, width, height)
    # Safety clipping: tiny numerical overshoots are okay; large overshoots mean bad extraction.
    if np.any(~np.isfinite(uv_norm)):
        return None, {"reason": "nonfinite_norm", "method": method}
    if np.max(np.abs(uv_norm)) > 1.25:
        return None, {"reason": "norm_out_of_range", "method": method, "max_abs": float(np.max(np.abs(uv_norm)))}

    out = {
        "uv": uv.astype(np.float32),
        "uv_norm": np.clip(uv_norm, -1.0, 1.0).astype(np.float32),
        "width": width,
        "height": height,
        "image_path": str(img_path) if img_path is not None else None,
        "image_key": img_key,
        "mask_path": str(mask_path) if mask_path is not None else None,
        "mask_key": mask_key,
        "method": method,
    }
    return out, meta


In [ ]:
# =========================
# Extract trajectories for selected rows
# =========================

trajectory_rows = []
skip_reasons = Counter()

t0 = time.time()

for i, row in enumerate(selected_rows):
    source = row["_source_dataset"]
    dataset_root = Path(row["_dataset_root"])

    traj, meta = extract_trajectory_for_row(row, dataset_root, m=M_POINTS)

    if traj is None:
        skip_reasons[meta.get("reason", "unknown")] += 1
    else:
        sample_id = row.get("id") or f"{source}_{i:08d}"
        out_row = {
            "id": f"{source.replace('-', '').replace(' ', '')}__{sample_id}",
            "original_id": sample_id,
            "source_dataset": source,
            "dataset_root": str(dataset_root),
            "split": row.get("split", row.get("_manifest_split", "unknown")),
            "sequence": row.get("sequence") or row.get("segment") or row.get("context_name"),
            "image_path": row.get("image_path") or row.get("rgb_path") or row.get("camera_image_path"),
            "mask_path": row.get(traj["mask_key"]) if traj.get("mask_key") else None,
            "resolved_image_path": traj["image_path"],
            "resolved_mask_path": traj["mask_path"],
            "image_width": traj["width"],
            "image_height": traj["height"],
            "trajectory_uv": traj["uv"].round(3).tolist(),
            "trajectory_uv_norm": traj["uv_norm"].round(6).tolist(),
            "trajectory_flat_norm": traj["uv_norm"].reshape(-1).round(6).tolist(),
            "num_points": M_POINTS,
            "extraction_method": traj["method"],
            "original": row,
        }
        trajectory_rows.append(out_row)

    if (i + 1) % 1000 == 0 or (i + 1) == len(selected_rows):
        print(f"{i+1:6d}/{len(selected_rows)} processed | extracted={len(trajectory_rows)} | skipped={sum(skip_reasons.values())}")

elapsed = time.time() - t0

print("\nDONE")
print("elapsed_sec:", round(elapsed, 2))
print("extracted:", len(trajectory_rows))
print("skipped:", sum(skip_reasons.values()))
print("skip_reasons:", dict(skip_reasons))
print("source counts:", Counter(r["source_dataset"] for r in trajectory_rows))
print("split counts:", Counter(r["split"] for r in trajectory_rows))

TRAJ_JSONL = OUT_DIR / "trajectory_vectors_combined_train.jsonl"
write_jsonl(TRAJ_JSONL, trajectory_rows)

SKIP_PATH = OUT_DIR / "trajectory_extraction_skip_reasons.json"
write_json(SKIP_PATH, {
    "skip_reasons": dict(skip_reasons),
    "selected_count": len(selected_rows),
    "extracted_count": len(trajectory_rows),
    "elapsed_sec": elapsed,
})

print("saved:", TRAJ_JSONL)
print("saved:", SKIP_PATH)


In [ ]:
# =========================
# Visual sanity check: extracted trajectories
# =========================

def draw_trajectory_on_image(row, max_w=520):
    img_path = Path(row["resolved_image_path"])
    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    pts = [(float(x), float(y)) for x, y in row["trajectory_uv"]]
    if len(pts) >= 2:
        draw.line(pts, fill=(255, 0, 0), width=10)
    for j, (x, y) in enumerate(pts):
        r = 6
        color = (0, 255, 0) if j == 0 else (0, 0, 255) if j == len(pts)-1 else (255, 255, 0)
        draw.ellipse((x-r, y-r, x+r, y+r), fill=color, outline=(0, 0, 0))

    label = f"{row['source_dataset']} | {row.get('sequence')} | {row['split']} | {row['extraction_method']}"
    draw.rectangle((8, 8, min(img.width-8, 1350), 46), fill=(255,255,255), outline=(0,0,0))
    draw.text((16, 18), label, fill=(0,0,0))

    w = max_w
    h = int(w * img.height / img.width)
    return img.resize((w, h))

# Balanced visual preview: a few Waymo and a few i2Nav
rng = random.Random(SEED)
waymo_vis = [r for r in trajectory_rows if r["source_dataset"] == "Waymo"]
i2nav_vis = [r for r in trajectory_rows if r["source_dataset"] == "i2Nav-Robot"]

selected_vis = []
if waymo_vis:
    selected_vis += rng.sample(waymo_vis, min(6, len(waymo_vis)))
if i2nav_vis:
    selected_vis += rng.sample(i2nav_vis, min(6, len(i2nav_vis)))

cards = [draw_trajectory_on_image(r, max_w=520) for r in selected_vis]

cols = 2
gap = 16
if cards:
    card_w, card_h = cards[0].size
    rows_grid = math.ceil(len(cards) / cols)
    grid = Image.new("RGB", (cols * card_w + (cols-1)*gap, rows_grid * card_h + (rows_grid-1)*gap), (245,245,245))
    for i, card in enumerate(cards):
        x = (i % cols) * (card_w + gap)
        y = (i // cols) * (card_h + gap)
        grid.paste(card, (x, y))
    display(grid)
    preview_path = OUT_DIR / "trajectory_extraction_preview.jpg"
    grid.save(preview_path, quality=95)
    print("saved:", preview_path)
else:
    print("No cards to display.")


In [ ]:
# =========================
# Prepare feature matrix for clustering
# =========================

X = np.asarray([r["trajectory_flat_norm"] for r in trajectory_rows], dtype=np.float32)
ids = [r["id"] for r in trajectory_rows]
sources = [r["source_dataset"] for r in trajectory_rows]

print("X shape:", X.shape)
print("finite:", np.isfinite(X).all())
print("min/max:", float(X.min()), float(X.max()))
print("source counts:", Counter(sources))

assert X.ndim == 2 and X.shape[1] == M_POINTS * 2
assert np.isfinite(X).all()


In [ ]:
# =========================
# Fit anchors for multiple K
# =========================

# We intentionally use OpenCV KMeans instead of sklearn KMeans.
# Reason: the current environment has sklearn installed, but sklearn import fails without scipy.
# cv2.kmeans is enough here because the feature matrix is simple: [N, 2*M] float32 trajectories.

def run_kmeans_cv2(X, K, attempts=8, max_iter=500, eps=1e-5, seed=42):
    """
    KMeans through OpenCV without sklearn/scipy.

    Args:
        X: np.ndarray [N, D], normalized trajectory vectors, float32/float64.
        K: number of clusters.
        attempts: number of independent initializations.
        max_iter: max iterations per attempt.
        eps: convergence epsilon.
        seed: OpenCV RNG seed.

    Returns:
        labels: np.ndarray [N] int64.
        centers: np.ndarray [K, D] float32.
        compactness: sum of squared distances to cluster centers.
        mse: compactness / N.
        rmse: sqrt(mse).
    """
    X32 = np.asarray(X, dtype=np.float32)
    assert X32.ndim == 2, X32.shape
    assert len(X32) >= K, f"Need at least K samples. len(X)={len(X32)}, K={K}"

    cv2.setRNGSeed(int(seed))

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        int(max_iter),
        float(eps),
    )

    compactness, labels, centers = cv2.kmeans(
        X32,
        int(K),
        None,
        criteria,
        int(attempts),
        cv2.KMEANS_PP_CENTERS,
    )

    labels = labels.reshape(-1).astype(np.int64)
    centers = centers.astype(np.float32)
    compactness = float(compactness)
    mse = compactness / max(len(X32), 1)
    rmse = float(np.sqrt(mse))

    return labels, centers, compactness, float(mse), rmse


cluster_results = []
models = {}

for K in K_LIST:
    print("\n" + "=" * 100)
    print("Fitting OpenCV KMeans K=", K)

    labels, centers, compactness, mse, rmse = run_kmeans_cv2(
        X,
        K=K,
        attempts=8,
        max_iter=500,
        eps=1e-5,
        seed=SEED,
    )

    counts_arr = np.bincount(labels, minlength=K)
    counts = {int(i): int(v) for i, v in enumerate(counts_arr.tolist())}
    min_count = int(counts_arr.min())
    max_count = int(counts_arr.max())
    empty_clusters = int((counts_arr == 0).sum())

    source_by_cluster = {}
    for c in range(K):
        srcs = [sources[i] for i in range(len(sources)) if labels[i] == c]
        source_by_cluster[str(c)] = dict(Counter(srcs))

    result = {
        "K": int(K),
        "algorithm": "cv2.kmeans",
        "attempts": 8,
        "max_iter": 500,
        "compactness": float(compactness),
        "mse": float(mse),
        "rmse": float(rmse),
        "inertia": float(compactness),
        "inertia_per_sample": float(mse),
        "silhouette_sample": None,
        "silhouette_note": "not computed: sklearn/scipy dependency intentionally avoided; use RMSE, cluster sizes, and visual anchor grids instead",
        "min_cluster_size": int(min_count),
        "max_cluster_size": int(max_count),
        "empty_clusters": int(empty_clusters),
        "cluster_sizes": {str(k): int(v) for k, v in sorted(counts.items())},
        "source_by_cluster": source_by_cluster,
    }

    cluster_results.append(result)
    models[K] = {
        "labels": labels,
        "centers": centers,
        "compactness": compactness,
        "mse": mse,
        "rmse": rmse,
    }

    print("compactness:", compactness)
    print("mse:", mse)
    print("rmse:", rmse)
    print("min/max cluster size:", min_count, max_count)
    print("empty_clusters:", empty_clusters)

RESULTS_PATH = OUT_DIR / "anchor_k_selection_metrics.json"
write_json(RESULTS_PATH, cluster_results)
print("\nsaved:", RESULTS_PATH)

print("\nK selection summary:")
for r in cluster_results:
    print(
        f"K={r['K']:2d} | mse={r['mse']:.6f} | rmse={r['rmse']:.6f} | "
        f"min_cluster={r['min_cluster_size']} | max_cluster={r['max_cluster_size']} | empty={r['empty_clusters']}"
    )


In [ ]:
# =========================
# Visualize anchors for each K
# =========================

def draw_anchor_grid(centers, K, canvas_w=640, canvas_h=384, cell_w=320, cell_h=220):
    anchors = centers.reshape(K, M_POINTS, 2)

    cols = 4
    rows_n = math.ceil(K / cols)
    grid = Image.new("RGB", (cols * cell_w, rows_n * cell_h), (245, 245, 245))

    for k in range(K):
        cell = Image.new("RGB", (cell_w, cell_h), (255, 255, 255))
        draw = ImageDraw.Draw(cell)

        # draw image plane frame
        margin = 26
        plot_w = cell_w - 2 * margin
        plot_h = cell_h - 2 * margin - 24
        x0, y0 = margin, margin + 18
        x1, y1 = x0 + plot_w, y0 + plot_h
        draw.rectangle((x0, y0, x1, y1), outline=(0, 0, 0), width=2)
        draw.text((10, 8), f"anchor {k}", fill=(0, 0, 0))

        uv = denormalize_uv(anchors[k], canvas_w, canvas_h)
        # map from canonical canvas to plot frame
        pts = []
        for x, y in uv:
            px = x0 + float(x) / max(canvas_w - 1, 1) * plot_w
            py = y0 + float(y) / max(canvas_h - 1, 1) * plot_h
            pts.append((px, py))

        if len(pts) >= 2:
            draw.line(pts, fill=(255, 0, 0), width=4)
        for j, (px, py) in enumerate(pts):
            r = 4
            color = (0, 180, 0) if j == 0 else (0, 0, 220) if j == len(pts)-1 else (240, 180, 0)
            draw.ellipse((px-r, py-r, px+r, py+r), fill=color, outline=(0, 0, 0))

        gx = (k % cols) * cell_w
        gy = (k // cols) * cell_h
        grid.paste(cell, (gx, gy))

    return grid

anchor_preview_paths = {}

for K in K_LIST:
    centers = models[K]["centers"]
    grid = draw_anchor_grid(centers, K)
    out_path = OUT_DIR / f"anchors_K{K:02d}_preview.jpg"
    grid.save(out_path, quality=95)
    anchor_preview_paths[K] = out_path
    print("saved:", out_path)

# Display main K=20
main_grid = Image.open(anchor_preview_paths[MAIN_K])
display(main_grid)


In [ ]:
# =========================
# Save main K=20 anchors and assignments
# =========================

MAIN_K = int(MAIN_K)
labels = models[MAIN_K]["labels"]
centers = models[MAIN_K]["centers"].astype(np.float32)
anchors = centers.reshape(MAIN_K, M_POINTS, 2)

ANCHORS_NPY = OUT_DIR / f"anchors_K{MAIN_K:02d}_M{M_POINTS}_norm.npy"
np.save(ANCHORS_NPY, anchors)

ANCHORS_JSON = OUT_DIR / f"anchors_K{MAIN_K:02d}_M{M_POINTS}_norm.json"
write_json(ANCHORS_JSON, {
    "K": MAIN_K,
    "M_POINTS": M_POINTS,
    "coordinate_system": "normalized image coordinates [-1, 1]",
    "shape": list(anchors.shape),
    "anchors": anchors.round(6).tolist(),
})

assignment_rows = []
for r, label in zip(trajectory_rows, labels.tolist()):
    rr = dict(r)
    rr["anchor_K"] = MAIN_K
    rr["anchor_id"] = int(label)
    rr["anchor_norm"] = anchors[int(label)].round(6).tolist()
    rr["anchor_distance_l2"] = float(np.linalg.norm(np.asarray(rr["trajectory_flat_norm"], dtype=np.float32) - centers[int(label)]))
    assignment_rows.append(rr)

ASSIGNMENTS_JSONL = OUT_DIR / f"trajectory_vectors_with_anchor_K{MAIN_K:02d}.jsonl"
write_jsonl(ASSIGNMENTS_JSONL, assignment_rows)

summary = {
    "project": "Cross-Domain Anchor Diffusion Planner / CDAD-Planner",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "project_root": str(PROJECT_ROOT),
    "waymo_root": str(WAYMO_ROOT),
    "i2nav_root": str(I2NAV_ROOT),
    "out_dir": str(OUT_DIR),
    "m_points": M_POINTS,
    "main_k": MAIN_K,
    "k_list": K_LIST,
    "selected_waymo": len(waymo_selected),
    "selected_i2nav": len(i2nav_selected),
    "extracted_trajectories": len(trajectory_rows),
    "source_counts": dict(Counter([r["source_dataset"] for r in trajectory_rows])),
    "skip_reasons": dict(skip_reasons),
    "anchor_files": {
        "anchors_npy": str(ANCHORS_NPY),
        "anchors_json": str(ANCHORS_JSON),
        "assignments_jsonl": str(ASSIGNMENTS_JSONL),
    },
    "k_selection_metrics": str(RESULTS_PATH),
    "anchor_preview_paths": {str(k): str(v) for k, v in anchor_preview_paths.items()},
    "kmeans_backend": "cv2.kmeans",
    "kmeans_note": "sklearn/scipy avoided intentionally; K selection uses MSE/RMSE, cluster sizes, domain distribution, and visual anchor previews",
    "method_note": "Trajectories are normalized image-space polylines extracted from existing corridor/centerline labels. These anchors are intended as priors for anchor-conditioned diffusion/flow refinement.",
}

SUMMARY_JSON = OUT_DIR / "anchor_generation_summary.json"
write_json(SUMMARY_JSON, summary)

print("saved anchors:", ANCHORS_NPY)
print("saved anchors json:", ANCHORS_JSON)
print("saved assignments:", ASSIGNMENTS_JSONL)
print("saved summary:", SUMMARY_JSON)
print("assignment rows:", len(assignment_rows))
print("anchor counts:", Counter([r["anchor_id"] for r in assignment_rows]))


In [ ]:
# =========================
# Source/domain distribution per K=20 anchor
# =========================

domain_table = []
for c in range(MAIN_K):
    rows_c = [r for r in assignment_rows if r["anchor_id"] == c]
    cnt = Counter(r["source_dataset"] for r in rows_c)
    seq = Counter(r.get("sequence") for r in rows_c)
    dist = [r["anchor_distance_l2"] for r in rows_c]
    domain_table.append({
        "anchor_id": c,
        "total": len(rows_c),
        "waymo": cnt.get("Waymo", 0),
        "i2nav": cnt.get("i2Nav-Robot", 0),
        "mean_l2": float(np.mean(dist)) if dist else None,
        "median_l2": float(np.median(dist)) if dist else None,
        "top_sequences": seq.most_common(5),
    })

DOMAIN_TABLE_JSON = OUT_DIR / f"anchor_K{MAIN_K:02d}_domain_distribution.json"
write_json(DOMAIN_TABLE_JSON, domain_table)

print("saved:", DOMAIN_TABLE_JSON)
for r in domain_table:
    print(
        f"anchor={r['anchor_id']:02d} total={r['total']:5d} "
        f"waymo={r['waymo']:5d} i2nav={r['i2nav']:5d} "
        f"mean_l2={r['mean_l2']:.4f}"
    )


## How to use the outputs for CDAD-Planner

Main files:

```text
cross_domain_anchor_trajectories_cdad_planner_v1/
  trajectory_vectors_combined_train.jsonl
  anchor_k_selection_metrics.json
  anchors_K20_M8_norm.npy
  anchors_K20_M8_norm.json
  trajectory_vectors_with_anchor_K20.jsonl
  anchor_K20_domain_distribution.json
  anchors_K20_preview.jpg
  anchor_generation_summary.json
```

Recommended model setup:

```text
Input:
  single RGB image

Target:
  normalized image-space trajectory, shape [M=8, 2]

Prior:
  one of K=20 shared cross-domain anchors

Model:
  image encoder
  anchor classifier
  conditional diffusion / flow refiner

Inference:
  image → top-k anchors → 1–5 refinement steps → final trajectory
```

For the article, justify `K=20` by:

1. showing K-selection metrics for `K = 8, 12, 16, 20, 24, 32`;
2. showing anchor preview grids;
3. reporting `MSE/RMSE` decrease as K increases;
4. showing that `K=20` gives enough maneuver diversity without over-fragmenting clusters;
5. reporting cluster sizes and domain distribution per anchor.

This notebook uses `cv2.kmeans`, not `sklearn.KMeans`. Therefore silhouette is intentionally not computed. This is acceptable for this stage: the anchor choice should be justified with reconstruction error, visual coverage of trajectory modes, cluster sizes, and cross-domain distribution.
